In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf
from IPython.display import display, HTML

# ============================================================
# DiD regression table: matched prices + matched population
# ============================================================

clean = Path("data/clean")
results = Path("results")
results.mkdir(exist_ok=True)

# -----------------------------
# Load data
# -----------------------------

prices_wide = pd.read_csv(clean / "matched_houses_population_2015_2024_prices.csv")
population_wide = pd.read_csv(clean / "matched_houses_population_2015_2024_population.csv")

treated = pd.read_csv(clean / "daniel_bowen_treated_suburbs_2022.csv")
untreated = pd.read_csv(clean / "daniel_bowen_untreated_suburbs_2022.csv")

# -----------------------------
# Convert wide files to long panel format
# -----------------------------

def wide_to_long(df, value_name):
    df = df.copy()
    df["suburb"] = df["suburb"].astype(str).str.strip().str.upper()

    out = df.melt(
        id_vars="suburb",
        var_name="year",
        value_name=value_name
    )

    out["year"] = out["year"].astype(int)
    return out

prices_long = wide_to_long(prices_wide, "price")
population_long = wide_to_long(population_wide, "population")

# -----------------------------
# Treatment data: all listed treated + untreated suburbs
# -----------------------------

treated = treated.copy()
untreated = untreated.copy()

treated["suburb"] = treated["suburb"].astype(str).str.strip().str.upper()
untreated["suburb"] = untreated["suburb"].astype(str).str.strip().str.upper()

treatment = pd.concat(
    [
        treated[["suburb", "treated"]],
        untreated[["suburb", "treated"]],
    ],
    ignore_index=True
)

treatment["treated"] = treatment["treated"].astype(int)
treatment = treatment.drop_duplicates(subset=["suburb"])

# -----------------------------
# Merge panel data
# -----------------------------

df_all = (
    treatment
    .merge(prices_long, on="suburb", how="left")
    .merge(population_long, on=["suburb", "year"], how="left")
)

missing_price_suburbs = sorted(
    df_all.loc[df_all["price"].isna(), "suburb"].dropna().unique()
)

df = df_all.dropna(subset=["price", "population", "treated"]).copy()

df = df[
    (df["price"] > 0) &
    (df["population"] > 0)
].copy()

# -----------------------------
# DiD variables
# -----------------------------

df["post"] = (df["year"] >= 2022).astype(int)
df["treated_post"] = df["treated"] * df["post"]

df["log_price"] = np.log(df["price"])
df["log_population"] = np.log(df["population"])

# -----------------------------
# Sample check
# -----------------------------

print("Final DiD sample")
print("----------------")
print("Listed suburbs:", treatment["suburb"].nunique())
print("Suburbs used in regression:", df["suburb"].nunique())
print("Observations:", len(df))
print("Years:", df["year"].min(), "-", df["year"].max())
print("Treated suburbs:", df.loc[df["treated"] == 1, "suburb"].nunique())
print("Control suburbs:", df.loc[df["treated"] == 0, "suburb"].nunique())

if len(missing_price_suburbs) > 0:
    print()
    print("Listed suburbs missing price/population data and therefore excluded:")
    for s in missing_price_suburbs:
        print("-", s)

print()
print("Observations by treatment and post period:")
print(df.groupby(["treated", "post"]).size())

# -----------------------------
# Preferred DiD regression
# Suburb FE and Year FE included
# -----------------------------

model = smf.ols(
    "log_price ~ treated_post + log_population + C(suburb) + C(year)",
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["suburb"]}
)

# -----------------------------
# Helper function for table rows
# -----------------------------

def model_row(label, var):
    if var not in model.params.index:
        return {
            "Variable": label,
            "Coef": "Absorbed",
            "Std Err": "",
            "t": "",
            "p-value": "",
            "CI": ""
        }

    coef = model.params[var]
    se = model.bse[var]
    t = model.tvalues[var]
    p = model.pvalues[var]
    ci_low, ci_high = model.conf_int().loc[var]

    return {
        "Variable": label,
        "Coef": f"{coef:.3f}",
        "Std Err": f"{se:.3f}",
        "t": f"{t:.3f}",
        "p-value": f"{p:.3f}",
        "CI": f"[{ci_low:.3f}, {ci_high:.3f}]"
    }

# -----------------------------
# Regression table in requested structure
# -----------------------------

regression_table = pd.DataFrame([
    model_row("Intercept", "Intercept"),

    # In a two-way fixed effects model, Treat is absorbed by suburb FE.
    {
        "Variable": "Treat",
        "Coef": "Absorbed by suburb FE",
        "Std Err": "",
        "t": "",
        "p-value": "",
        "CI": ""
    },

    # In a two-way fixed effects model, Post is absorbed by year FE.
    {
        "Variable": "Post",
        "Coef": "Absorbed by year FE",
        "Std Err": "",
        "t": "",
        "p-value": "",
        "CI": ""
    },

    model_row("Treat x Post (DiD effect)", "treated_post"),
    model_row("log_population", "log_population"),

    {
        "Variable": "Suburb FE",
        "Coef": "Yes",
        "Std Err": "",
        "t": "",
        "p-value": "",
        "CI": ""
    },
    {
        "Variable": "Year FE",
        "Coef": "Yes",
        "Std Err": "",
        "t": "",
        "p-value": "",
        "CI": ""
    },
    {
        "Variable": "N",
        "Coef": f"{int(model.nobs):,}",
        "Std Err": "",
        "t": "",
        "p-value": "",
        "CI": ""
    },
    {
        "Variable": "R²",
        "Coef": f"{model.rsquared:.3f}",
        "Std Err": "",
        "t": "",
        "p-value": "",
        "CI": ""
    }
])

# -----------------------------
# Display styled table
# -----------------------------

styled_html = regression_table.to_html(index=False, escape=False)

styled_html = styled_html.replace(
    "<table border=\"1\" class=\"dataframe\">",
    """
    <table style="
        border-collapse: collapse;
        width: 95%;
        font-family: Arial, sans-serif;
        font-size: 15px;
        border: 1px solid #d8c7b6;
    ">
    """
)

styled_html = styled_html.replace(
    "<th>",
    '<th style="background-color:#f7eee5; padding:12px; text-align:left; border:1px solid #d8c7b6;">'
)

styled_html = styled_html.replace(
    "<td>",
    '<td style="padding:11px; text-align:left; border:1px solid #d8c7b6;">'
)

display(HTML(styled_html))

# -----------------------------
# Save table
# -----------------------------

regression_table.to_csv(results / "did_regression_table_final.csv", index=False)

with open(results / "did_regression_table_final.html", "w", encoding="utf-8") as f:
    f.write(styled_html)

print()
print("Saved regression table to:")
print(results / "did_regression_table_final.csv")
print(results / "did_regression_table_final.html")

# -----------------------------
# Interpretation
# -----------------------------

did_coef = model.params["treated_post"]
did_se = model.bse["treated_post"]
did_pct = (np.exp(did_coef) - 1) * 100

print()
print("Main DiD result")
print("---------------")
print(f"Treat x Post coefficient: {did_coef:.3f}")
print(f"Clustered standard error: {did_se:.3f}")
print(f"Approximate percentage effect: {did_pct:.1f}%")
print()
print(
    "Interpretation: after 2022, treated suburbs experienced an estimated "
    f"{did_pct:.1f}% relative change in median house prices compared with control suburbs, "
    "holding population constant and including suburb and year fixed effects."
)


Final DiD sample
----------------
Listed suburbs: 179
Suburbs used in regression: 49
Observations: 490
Years: 2015.0 - 2024.0
Treated suburbs: 24
Control suburbs: 25

Listed suburbs missing price/population data and therefore excluded:
- ALPHINGTON
- ARDEER
- ASPENDALE
- AVENEL
- BALLARAT
- BAXTER
- BEACONSFIELD
- BELL PARK
- BENTLEIGH
- BERWICK
- BITTERN
- BLACKBURN
- BONBEACH
- BOWSER
- BRIGHTON
- BROOKLYN
- BRUNSWICK
- BUNYIP
- BURNLEY
- CALDER PARK
- CARRUM
- CAULFIELD
- CHELSEA
- CHELTENHAM
- CLAYTON
- CLIFTON HILL
- COBURG
- CORIO
- CRANBOURNE NORTH
- CRESSY
- DANDENONG
- DANDENONG SOUTH
- DIAMOND CREEK
- DIMBOOLA
- DONNYBROOK
- EDITHVALE
- ELTHAM
- EPPING
- ESSENDON
- EUROA
- FAIRFIELD
- FAWKNER
- FERNTREE GULLY
- GLEN IRIS
- GLENHUNTLY
- GLENORCHY
- GLENROY
- GROVEDALE
- HASTINGS
- HEATHCOTE JUNCTION
- HIGHETT
- HOPPERS CROSSING
- HUGHESDALE
- IVANHOE
- KANIVA
- KENSINGTON
- KEON PARK
- KOOYONG
- LILYDALE
- LITTLE RIVER
- LOCKSLEY
- LONGWARRY
- LONGWOOD
- LYNDHURST
- MACLEOD
- 

Variable,Coef,Std Err,t,p-value,CI
Intercept,10.662,0.655,16.282,0.000,"[9.378, 11.945]"
Treat,Absorbed by suburb FE,,,,
Post,Absorbed by year FE,,,,
Treat x Post (DiD effect),-0.075,0.030,-2.484,0.013,"[-0.134, -0.016]"
log_population,0.300,0.069,4.327,0.000,"[0.164, 0.436]"
Suburb FE,Yes,,,,
Year FE,Yes,,,,
N,490,,,,
R²,0.983,,,,



Saved regression table to:
results\did_regression_table_final.csv
results\did_regression_table_final.html

Main DiD result
---------------
Treat x Post coefficient: -0.075
Clustered standard error: 0.030
Approximate percentage effect: -7.2%

Interpretation: after 2022, treated suburbs experienced an estimated -7.2% relative change in median house prices compared with control suburbs, holding population constant and including suburb and year fixed effects.


Key thing: because this includes Suburb FE and Year FE, Treat and Post are not separately estimated. That is correct. Treat is absorbed by suburb fixed effects, and Post is absorbed by year fixed effects. The DiD effect is still estimated by Treat x Post.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf
from IPython.display import display, HTML

clean = Path("data/clean")
results = Path("results")
results.mkdir(exist_ok=True)

# -----------------------------
# Load data
# -----------------------------

treated_file = pd.read_csv(clean / "daniel_bowen_treated_suburbs_2022.csv")
untreated_file = pd.read_csv(clean / "daniel_bowen_untreated_suburbs_2022.csv")
prices_wide = pd.read_csv(clean / "suburbs_house_prices_2015_2024_wide_complete.csv")
population_long = pd.read_csv(clean / "SA2_2021_AUST_clean.csv")

# -----------------------------
# Treatment data
# -----------------------------

treated_file["suburb"] = treated_file["suburb"].astype(str).str.strip().str.upper()
untreated_file["suburb"] = untreated_file["suburb"].astype(str).str.strip().str.upper()

treatment = pd.concat(
    [
        treated_file[["suburb", "treated"]],
        untreated_file[["suburb", "treated"]],
    ],
    ignore_index=True
)

treatment["treated"] = treatment["treated"].astype(int)
treatment = treatment.drop_duplicates("suburb")

# -----------------------------
# Price data
# -----------------------------

prices_wide["Suburb"] = prices_wide["Suburb"].astype(str).str.strip().str.upper()

prices_long = prices_wide.melt(
    id_vars="Suburb",
    var_name="year",
    value_name="price"
)

prices_long = prices_long.rename(columns={"Suburb": "suburb"})
prices_long["year"] = prices_long["year"].astype(int)

# -----------------------------
# Population data
# -----------------------------

population_long = population_long.rename(columns={
    "Suburb": "suburb",
    "Year": "year",
    "Population": "population"
})

population_long["suburb"] = population_long["suburb"].astype(str).str.strip().str.upper()
population_long["year"] = population_long["year"].astype(int)
population_long = population_long[["suburb", "year", "population"]]

# -----------------------------
# Merge data
# -----------------------------

df = (
    treatment
    .merge(prices_long, on="suburb", how="left")
    .merge(population_long, on=["suburb", "year"], how="left")
    .dropna(subset=["price", "population", "treated"])
    .copy()
)

df = df[(df["price"] > 0) & (df["population"] > 0)].copy()

# -----------------------------
# DiD variables
# -----------------------------

df["post"] = (df["year"] >= 2022).astype(int)
df["treat_post"] = df["treated"] * df["post"]

df["log_price"] = np.log(df["price"])
df["log_population"] = np.log(df["population"])

# -----------------------------
# Regression with fixed effects
# -----------------------------

model = smf.ols(
    "log_price ~ treat_post + log_population + C(suburb) + C(year)",
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["suburb"]}
)

# -----------------------------
# Create table rows
# -----------------------------

def get_row(label, variable):
    if variable not in model.params.index:
        return [label, "Absorbed", "", "", "", ""]

    coef = model.params[variable]
    se = model.bse[variable]
    t_val = model.tvalues[variable]
    p_val = model.pvalues[variable]
    ci_low, ci_high = model.conf_int().loc[variable]

    return [
        label,
        f"{coef:.3f}",
        f"{se:.3f}",
        f"{t_val:.3f}",
        f"{p_val:.3f}",
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    ]

did_table = pd.DataFrame(
    [
        get_row("Intercept", "Intercept"),
        ["Treat", "Absorbed by suburb FE", "", "", "", ""],
        ["Post", "Absorbed by year FE", "", "", "", ""],
        get_row("Treat × Post (DiD effect)", "treat_post"),
        get_row("log_population", "log_population"),
        ["Suburb FE", "Yes", "", "", "", ""],
        ["Year FE", "Yes", "", "", "", ""],
        ["N", f"{int(model.nobs):,}", "", "", "", ""],
        ["R²", f"{model.rsquared:.3f}", "", "", "", ""],
    ],
    columns=["Variable", "Coef", "Std Err", "t", "p-value", "CI"]
)

# -----------------------------
# Display formatted table
# -----------------------------

html = did_table.to_html(index=False, escape=False)

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:separate; border-spacing:0; width:92%; font-family:Arial, sans-serif; font-size:16px; color:#2b2927; border:1px solid #dcc7b3; border-radius:10px; overflow:hidden;">'
)

html = html.replace(
    "<thead>",
    '<thead style="background-color:#f7eee5;">'
)

html = html.replace(
    "<th>",
    '<th style="padding:14px 18px; text-align:left; font-weight:700; border-right:1px solid #dcc7b3; border-bottom:1px solid #dcc7b3;">'
)

html = html.replace(
    "<td>",
    '<td style="padding:14px 18px; text-align:left; border-right:1px solid #dcc7b3; border-bottom:1px solid #dcc7b3;">'
)

display(HTML(html))

# -----------------------------
# Save outputs
# -----------------------------

did_table.to_csv(results / "did_regression_table_formatted.csv", index=False)

with open(results / "did_regression_table_formatted.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Saved:")
print(results / "did_regression_table_formatted.csv")
print(results / "did_regression_table_formatted.html")

# -----------------------------
# Interpretation
# -----------------------------

did_coef = model.params["treat_post"]
did_se = model.bse["treat_post"]
did_p = model.pvalues["treat_post"]
did_pct = (np.exp(did_coef) - 1) * 100

print()
print("Interpretation")
print("--------------")
print(f"Treat × Post coefficient: {did_coef:.3f}")
print(f"Clustered standard error: {did_se:.3f}")
print(f"p-value: {did_p:.3f}")
print(f"Approximate percentage effect: {did_pct:.1f}%")
print()
print(
    "After 2022, treated suburbs experienced an estimated "
    f"{did_pct:.1f}% relative change in median house prices compared with untreated suburbs, "
    "holding population constant and controlling for suburb and year fixed effects."
)


Variable,Coef,Std Err,t,p-value,CI
Intercept,12.448,0.383,32.476,0.000,"[11.697, 13.200]"
Treat,Absorbed by suburb FE,,,,
Post,Absorbed by year FE,,,,
Treat × Post (DiD effect),0.305,0.019,16.045,0.000,"[0.268, 0.342]"
log_population,0.180,0.043,4.214,0.000,"[0.096, 0.264]"
Suburb FE,Yes,,,,
Year FE,Yes,,,,
N,250,,,,
R²,0.984,,,,


Saved:
results\did_regression_table_formatted.csv
results\did_regression_table_formatted.html

Interpretation
--------------
Treat × Post coefficient: 0.305
Clustered standard error: 0.019
p-value: 0.000
Approximate percentage effect: 35.7%

After 2022, treated suburbs experienced an estimated 35.7% relative change in median house prices compared with untreated suburbs, holding population constant and controlling for suburb and year fixed effects.


The Daniel Bowen files do not contain price or population. They only contain:

suburb, status, treated
So for a DiD regression, we must use:

Daniel Bowen files = treatment/control suburbs
suburbs_house_prices_2015_2024_wide_complete.csv = prices
SA2_2021_AUST_clean.csv = population